# Xiangqi AI — Bronze Dataset Ingestion

Notebook chỉ **ingest dữ liệu thô bất biến** về Google Drive, có SHA-256/manifest và chống ghi đè. Không replay, chuyển định dạng, dedup ván hay train; các bước đó là nhiệm vụ Silver/Gold.

Dữ liệu Alpha-Beta có sẵn tại `MyDrive/XiangqiAI/datasets`; Bronze nằm trong `MyDrive/XiangqiAI/data/01_bronze`. Trường hợp CCPD hoặc nguồn khác, tải file hợp pháp về thư mục input trước, rồi cấu hình `SOURCE` và `INCLUDE_PATTERNS`. License phải được tự xác nhận.

## 1. Mount Google Drive

Chạy trên Colab. Drive sẽ yêu cầu quyền truy cập từ bạn.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_PROJECT = Path("/content/drive/MyDrive/XiangqiAI")
INPUT_DIR = DRIVE_PROJECT / "datasets"
BRONZE_ROOT = DRIVE_PROJECT / "data" / "01_bronze"
REPO_DIR = Path("/content/ML-Advance-Group")
BACKEND_DIR = REPO_DIR / "project" / "backend"

SOURCE = "alphabeta"
SOURCE_VERSION = "colab-batch-v1"
INCLUDE_PATTERNS = ["*.jsonl"]
SOURCE_LICENSE = None
BRANCH = "main"

if not INPUT_DIR.is_dir():
    raise FileNotFoundError(f"Input directory not found: {INPUT_DIR}")
print("Input:", INPUT_DIR)
print("Bronze:", BRONZE_ROOT / SOURCE)

## 2. Checkout backend từ GitHub

Dùng mã `BronzeIngester` trong `src/data/01_bronze/data_pipeline/`, không viết lại logic ingest trong notebook.

In [ ]:
import subprocess
import sys

if not REPO_DIR.exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", BRANCH,
        "https://github.com/Ko4rut/ML-Advance-Group.git", str(REPO_DIR),
    ], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "switch", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(BACKEND_DIR)],
    check=True,
)
assert (BACKEND_DIR / "scripts" / "ingest_bronze.py").is_file()
print("Backend:", BACKEND_DIR)

## 3. Kiểm tra nguồn

Chỉ nhận các định dạng được chọn. Theo mặc định `*.jsonl` sẽ loại checkpoint `.json`, các file trống được ghi nhận trong báo cáo nhưng không tạo bản Bronze. Với nguồn khác, đổi `INCLUDE_PATTERNS` theo đuôi thực tế (ví dụ `*.pgn`, `*.txt`); không tự giả định license.

In [ ]:
from fnmatch import fnmatch

files = sorted(
    p for p in INPUT_DIR.rglob("*")
    if p.is_file() and any(fnmatch(p.name, pattern) for pattern in INCLUDE_PATTERNS)
)
print("Discovered:", len(files))
for p in files[:30]:
    print(p.relative_to(INPUT_DIR), f"({p.stat().st_size:,} bytes)")
if not files:
    raise ValueError("No files matched INCLUDE_PATTERNS")

## 4. Chạy ingest Bronze (idempotent)

File đã có cùng SHA-256 sẽ được bỏ qua; nếu cùng tên nhưng khác nội dung, chương trình **dừng** để bảo vệ dữ liệu cũ. Với một dataset mới, có thể nhập cả folder; thư mục con được giữ nguyên.

In [ ]:
import json

command = [
    sys.executable, "-m", "scripts.ingest_bronze",
    str(INPUT_DIR),
    "--destination-root", str(BRONZE_ROOT),
    "--source", SOURCE,
    "--source-version", SOURCE_VERSION,
    "--include", *INCLUDE_PATTERNS,
]
if SOURCE_LICENSE is not None:
    command.extend(["--license", SOURCE_LICENSE])

subprocess.run(command, cwd=BACKEND_DIR, check=True)

## 5. Đối chiếu checksum bản Bronze

Đọc manifest tương ứng với file, kiểm tra hash và kích thước. Với các file tồn tại sẵn, manifest tạo bằng notebook Bronze cũ vẫn được chấp nhận.

In [ ]:
import hashlib
from src.data.bronze.models import BronzeFileManifest

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

verified = 0
for original in files:
    if original.stat().st_size == 0:
        continue
    relative = original.relative_to(INPUT_DIR)
    copied = BRONZE_ROOT / SOURCE / relative
    sidecar = copied.with_name(copied.name + ".manifest.json")
    manifest = BronzeFileManifest.model_validate_json(sidecar.read_text(encoding="utf-8"))
    assert copied.is_file()
    assert manifest.file_sha256 == sha256_file(original) == sha256_file(copied)
    assert manifest.source == SOURCE
    verified += 1
print("Verified", verified, "non-empty files")
print("Bronze directory:", BRONZE_ROOT / SOURCE)

## Nguồn bên ngoài / CCPD

Notebook này ingest từ thư mục đã có dữ liệu. Để thêm CCPD, tải trước file theo đúng giấy phép/phương thức phân phối từ tác giả vào một thư mục riêng trong Drive (hoặc `/content/`), rồi đổi `INPUT_DIR`, `SOURCE='ccpd'`, `SOURCE_VERSION`, `INCLUDE_PATTERNS`, `SOURCE_LICENSE` trước khi chạy cell 3–5. Không tự biến file CCPD thành Alpha-Beta JSONL. Việc giải mã notation nằm ở Silver Adapter.